In [ ]:
import sys
from datetime import date
from pathlib import Path

# Asegura que la carpeta del proyecto (donde está `config/` y `src/`) esté en sys.path
# Ajusta parents[...] si tu notebook está en una subcarpeta distinta dentro de `files/`
notebook_dir = Path.cwd()
for candidate in [notebook_dir, *notebook_dir.parents]:
    if (candidate / "config" / "secrets.py").exists() and (candidate / "src" / "bronze").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.bronze.aemet_volume_ingestor import (
    DEFAULT_OUTPUT_FILE,
    ingest_aemet_historical_2015_today,
)

# Cambia TEST_MODE a False cuando quieras descargar todo el histórico 2015 -> hoy
TEST_MODE = False

if TEST_MODE:
    # Prueba corta (últimos 30 días) para verificar credenciales, API de AEMET y escritura en Volume
    summary = ingest_aemet_historical_2015_today(
        output_file="/Volumes/workspace/default/bronze_aemet/diarios-extremadura-test.jsonl",
        date_from="2026-10-25",
        date_to=date.today().isoformat(),
        window_days=31,
        sleep_seconds=1.0,
        reset_file=True,
    )
else:
    # Histórico completo de 11 años (2015-01-01 hasta hoy) para Extremadura
    summary = ingest_aemet_historical_2015_today(
        output_file=DEFAULT_OUTPUT_FILE,
        date_from="2015-01-01",
        date_to=date.today().isoformat(),
        window_days=180,
        sleep_seconds=1.2,
        reset_file=True,
    )

print("Resumen ingesta Bronze AEMET:", summary)

In [ ]:
from pyspark.sql import functions as F

df_bronze = spark.read.json("/Volumes/workspace/default/bronze_aemet/data-2015-2026.jsonl")

# Filtrar por provincia y ver cuántas filas hay por año
df_bronze.filter(F.col("provincia") == "BADAJOZ") \
    .withColumn("anio", F.substring("fecha", 1, 4)) \
    .groupBy("anio") \
    .count() \
    .orderBy("anio") \
    .show(20, truncate=False)

In [ ]:
import sys
from pathlib import Path

ROOT = str(Path.cwd().resolve().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.silver.aemet_cleaner import (
    DEFAULT_BRONZE_PATH,
    DEFAULT_SILVER_TABLE,
    build_silver_aemet,
)

res_silver = build_silver_aemet(
    spark=spark,
    bronze_path=DEFAULT_BRONZE_PATH,
    silver_table=DEFAULT_SILVER_TABLE,
)
print("Silver generada:", res_silver)

In [ ]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.silver_aemet_diarios_extremadura")

# 1. Resumen global: filas, clave única, estaciones y fechas
resumen = df.agg(
    F.count("*").alias("total_filas"),
    F.countDistinct("indicativo", "fecha_dt").alias("claves_unicas"),
    F.countDistinct("indicativo").alias("estaciones"),
    F.min("fecha_dt").alias("fecha_min"),
    F.max("fecha_dt").alias("fecha_max"),
    F.sum(F.col("tmax_c").isNull().cast("int")).alias("nulos_tmax"),
    F.sum(F.col("prec_mm").isNull().cast("int")).alias("nulos_prec"),
    F.sum(F.col("hr_min_pct").isNull().cast("int")).alias("nulos_hr_min"),
    F.sum(F.col("dias_secos_consecutivos").isNull().cast("int")).alias("nulos_racha_seca"),
)
resumen.show(truncate=False)

# 2. Comprobar que total_filas == claves_unicas
row = resumen.first()
assert row["total_filas"] == row["claves_unicas"], "¡Hay duplicados por (indicativo, fecha_dt)!"
print("OK: Sin duplicados por (indicativo, fecha_dt).")